In [1]:
import pandas as pd
import numpy as np
import pickle, os
from recsys_utils import build_split, evaluate, popularity_recs, random_recs, METRIC_COLS

listings_clean = pd.read_pickle('../data/processed/listings_clean.pkl')
reviews_df = pd.read_csv('../data/Reviews.csv', encoding='latin1')
print(listings_clean.shape, reviews_df.shape)

(279599, 37) (5373143, 4)


Այստեղ կատարել եմ տվյալների բաժանումում ՝train և test բաժինների և ստուգել եմ որ չլինի data leakage ,որպեսզի մոդելը նախապես չտեսնի այն տվյալները որի վրա պետք է քննվի և գնահատվի,որը ապահովում է գնահատականների գիտական օբյեկտիվությունն ու հավաստիությունը

In [2]:
SPLIT_PATH = '../data/processed/split.pkl'
if os.path.exists(SPLIT_PATH):
    split = pickle.load(open(SPLIT_PATH, 'rb'))
else:
    split = build_split(reviews_df, listings_clean, min_interactions=3, n_eval_users=10000, seed=42)
    pickle.dump(split, open(SPLIT_PATH, 'wb'))

print("user", split['n_users'], "| houses:", split['n_items'])
print("Train interactions:", split['train'].nnz)
print("evaluation users:", len(split['eval_users']))

leaks = sum(1 for u, items in split['test_truth'].items()
            if split['trainval'][u, list(items)[0]] > 0)
assert leaks == 0, f"Leakage: {leaks}"
print("Leakage Check: OK")

user 140786 | houses: 279599
Train interactions: 248303
evaluation users: 10000
Leakage Check: OK


Առաջինը Popularity Baseline-ն է, որը տները տեսակավորում է ըստ կարծիքների քանակի և վարկանիշի՝ ընտրելով ամենահայտնի ու լավագույն գնահատված բնակարանները: Երկրորդը Random Baseline-ն է, որը random_state=42-ի միջոցով վերարտադրելիորեն ընտրում է պատահական տներ:Սրանք մեր հիմնական ելակետային շեմերն են, որոնց հետ հետագայում համեմատվելու են մեր կողմից մշակվող անհատականացված ռեկոմենդացիոն ալգորիթմները։

In [3]:
def get_baseline_popular_recommendations(listings_df, top_n=5):
    popular = listings_df.sort_values(by=['number_of_reviews', 'review_scores_rating'],
                                      ascending=False).head(top_n)
    cols = ['listing_id', 'name', 'city', 'room_type', 'price', 'number_of_reviews', 'review_scores_rating']
    return popular[[c for c in cols if c in popular.columns]]

def get_baseline_random_recommendations(listings_df, top_n=5, random_state=42):
    rnd = listings_df.sample(n=top_n, random_state=random_state)
    cols = ['listing_id', 'name', 'city', 'room_type', 'price']
    return rnd[[c for c in cols if c in rnd.columns]]

display(get_baseline_popular_recommendations(listings_clean))
display(get_baseline_random_recommendations(listings_clean))

,listing_id,name,city,room_type,price,number_of_reviews,review_scores_rating
167067,17222007,Sweet & cosy room next to Canal Saint Martin Ã...,Paris,Private room,76,891,94.0
131663,8637229,Ã©Â¦â¢Ã¦Â¸Â¯ Hong Kong Ã©â¢Â·Ã¦Â´Â² Cheung C...,Hong Kong,Private room,629,828,96.0
212518,1249964,A Journey Into The Heart Of Paris,Paris,Private room,81,796,97.0
146347,32011332,Bed in 8 Bed Dorm,Paris,Hotel room,27,762,87.0
142662,2399029,B&B Rione Monti,Rome,Private room,39,754,93.0


,listing_id,name,city,room_type,price
65097,28608931,DARLING HARBOUR 2 BEDROOM APARTMENT WITH CITY ...,Sydney,Entire place,307
207180,42252699,Guesthouse clodin roma,Rome,Private room,210
98085,32248883,A guest suite perfectly situated for the trave...,Cape Town,Entire place,450
181978,14128632,Double Furnished Bedroom in Mona Vale (Waterfr...,Sydney,Private room,55
195077,14963583,Room in South Harlem near Central Park,New York,Private room,70


Այստեղ գնահատել եմ երկու բազային մոդելները 10,000 թեստային օգտատերերի վրա

In [4]:
K = 5
users = split['eval_users']
seen = split['trainval']
pop = split['pop_trainval']

pop_recs  = popularity_recs(users, pop, seen, k=K)
rand_recs = random_recs(users, split['n_items'], seen, k=K, seed=42)

args = dict(item_pop=pop, n_users=split['n_users'], n_items=split['n_items'], k=K)
res_pop  = evaluate(pop_recs,  split['test_truth'], **args)
res_rand = evaluate(rand_recs, split['test_truth'], **args)

baseline_results = pd.DataFrame({'Random': res_rand, 'Popularity': res_pop}).T[METRIC_COLS]
baseline_results.index.name = 'Model'
display(baseline_results.round(5))

baseline_results.to_pickle('../data/processed/results_baselines.pkl')

,Precision@5,Recall@5,MAP@5,NDCG@5,Coverage,Novelty
Model,,,,,,
Random,0.0000,0.0000,0.00000,0.00000,0.16406,16.64623
Popularity,0.0001,0.0005,0.00021,0.00028,0.00002,10.84926
